# Lab 7 — Fase 4: Dois Canais (pseudo-binaural, ouvidos a 18 cm)
**Processamento de Sinais Multimédia** · 80–110 min do roteiro · plano horizontal ($z=0$)

**Objetivo da fase:** distribuir as reflexões lateralmente e gerar saída estéreo. Cada ouvido é um recetor a $\pm 9$ cm do centro da cabeça; para cada um calculam-se atrasos e ganhos da onda direta e das 4 reflexões, e a cauda reverberante é descorrelacionada entre canais.

**Entradas herdadas das fases anteriores** (célula 1): sala, fonte, ouvinte, $RT_{60}$, absorção e filtros *Comb/All-Pass* do TP1. Os valores de posições e $RT_{60}$ são assumidos; substituir pelos do grupo.

In [ ]:
import numpy as np
import wave
import matplotlib.pyplot as plt
from IPython.display import Audio, display
plt.rcParams['figure.figsize'] = (12, 4.5); plt.rcParams['axes.grid'] = True

# ===================== ENTRADAS DAS FASES ANTERIORES =====================
Fs, c = 44100, 343.0
Lx, Ly = 7.7, 9.4                       # sala (m)
SRC    = np.array([2.5, 2.0])           # fonte (x, y)          <-- assumida
LIS    = np.array([5.0, 6.5])           # centro da cabeça      <-- assumida
FACING = np.array([0.0, -1.0])          # o ouvinte olha para y = 0
RT60   = 0.80                           # s                     <-- PLACEHOLDER (usar o medido)
ALPHA  = 0.161 * Lx * Ly / (RT60 * 2 * (Lx + Ly))   # Sabine 2D
R      = np.sqrt(1 - ALPHA)             # amplitude refletida por parede
T_MIX  = 0.050                          # tempo de mistura (s)
DRR_DB = -6.0                           # energia direta / energia da cauda (dB)
A_AP   = 0.7
T_COMB_L, T_AP_L = (29.7, 37.1, 41.1, 43.7), (5.0, 1.7)   # filtros do TP1 (canal esq.)

# ===================== PARÂMETROS DESTA FASE =====================
HEAD   = 0.18                           # distância entre ouvidos (m)
BETA   = 0.35                           # sombra da cabeça: ganho = 1 ± BETA·cosθ
T_COMB_R, T_AP_R = (31.1, 35.3, 42.3, 45.1), (5.3, 1.9)   # canal dir.: atrasos distintos
AUDIO_IN  = '../../psm-tp1-49774-51811/entrega/audio/in/voz_seca.wav'
AUDIO_OUT = 'Lab7_audio'

In [ ]:
def read_wav(path):
    with wave.open(path) as w:
        ch = w.getnchannels()
        x = np.frombuffer(w.readframes(w.getnframes()), dtype=np.int16) / 32768.0
    return x.reshape(-1, ch).mean(axis=1) if ch > 1 else x

def write_wav(path, x):
    pcm = np.round(np.clip(x, -1, 1) * 32767).astype(np.int16)
    with wave.open(path, 'wb') as w:
        w.setnchannels(1 if x.ndim == 1 else x.shape[1]); w.setsampwidth(2); w.setframerate(Fs)
        w.writeframes(pcm.tobytes())

def fftconv(x, h):
    n = len(x) + len(h) - 1; N = 1 << (n - 1).bit_length()
    return np.fft.irfft(np.fft.rfft(x, N) * np.fft.rfft(h, N), N)[:n]

def recursive_delay(u, N, g):            # y[n] = u[n] + g·y[n-N]
    y = u.astype(float).copy()
    for s in range(N, len(y), N):
        e = min(s + N, len(y)); y[s:e] += g * y[s - N:e - N]
    return y

def comb(x, N, g):    return recursive_delay(x, N, g)
def allpass(x, N, a):
    v = -a * x; v[N:] += x[:-N]; return recursive_delay(v, N, a)

## 1. Geometria dos dois ouvidos
O vetor para a esquerda do ouvinte é `FACING` rodado $+90°$. Os ouvidos ficam a $\pm HEAD/2$ ao longo desse eixo. O atraso interaural máximo é $0{,}18/343 \approx 0{,}52$ ms $\approx 23$ amostras.

In [ ]:
f = FACING / np.linalg.norm(FACING)
left = np.array([-f[1], f[0]])
EAR_L = LIS + (HEAD / 2) * left
EAR_R = LIS - (HEAD / 2) * left
print('Ouvido esq.:', EAR_L.round(3), '  Ouvido dir.:', EAR_R.round(3), '  separação:', np.linalg.norm(EAR_L - EAR_R), 'm')
print(f'ITD máximo: {HEAD / c * 1000:.2f} ms = {HEAD / c * Fs:.1f} amostras')

## 2. Reflexões para cada ouvido (método das imagens + lateralização)
Para cada ouvido, $d_i$ é a distância à fonte-imagem de cada parede, $n_i = \mathrm{round}(d_i F_s/c)$ e

$$g_i = \frac{R}{d_i}\,\bigl(1 \pm \beta\cos\theta_i\bigr)$$

onde $\cos\theta_i$ é a componente da direção de chegada ao longo do eixo esquerda–direita ($+$ ouvido esquerdo, $-$ direito). Uma reflexão que chega pela esquerda reforça o canal esquerdo e atenua o direito.

In [ ]:
SURFACES = ['Parede x = 0', 'Parede x = Lx', 'Parede y = 0', 'Parede y = Ly']
IMAGES = [np.array(p) for p in ((-SRC[0], SRC[1]), (2 * Lx - SRC[0], SRC[1]),
                                (SRC[0], -SRC[1]), (SRC[0], 2 * Ly - SRC[1]))]

def taps(rx, side):
    out = []
    for name, pos, refl in [('Onda direta', SRC, 1.0)] + [(n, p, R) for n, p in zip(SURFACES, IMAGES)]:
        v = pos - rx; d = np.linalg.norm(v)
        lateral = np.dot(v / d, left)
        out.append(dict(nome=name, d=d, t_ms=1000 * d / c, n=int(round(d / c * Fs)),
                        g0=refl / d, g=refl / d * (1 + BETA * side * lateral)))
    return out

tab_L, tab_R = taps(EAR_L, +1), taps(EAR_R, -1)
for lado, tab in (('OUVIDO ESQUERDO', tab_L), ('OUVIDO DIREITO', tab_R)):
    print(lado)
    print(f"{'Superfície':<16}{'d (m)':>8}{'t (ms)':>9}{'n (amostras)':>14}{'ganho (R/d)':>13}{'ganho c/ cabeça':>17}")
    for r in tab:
        print(f"{r['nome']:<16}{r['d']:>8.3f}{r['t_ms']:>9.2f}{r['n']:>14d}{r['g0']:>13.4f}{r['g']:>17.4f}")
    print()
print('Diferença interaural (esq - dir) em amostras:', {a['nome']: a['n'] - b['n'] for a, b in zip(tab_L, tab_R)})

with open(f'{AUDIO_OUT}/tabela_reflexoes_estereo.csv', 'w', encoding='utf-8') as fh:
    fh.write('ouvido,superficie,distancia_m,atraso_ms,atraso_amostras,ganho_R_sobre_d,ganho_com_cabeca\n')
    for lado, tab in (('esq', tab_L), ('dir', tab_R)):
        for r in tab:
            fh.write(f"{lado},{r['nome']},{r['d']:.4f},{r['t_ms']:.3f},{r['n']},{r['g0']:.5f},{r['g']:.5f}\n")

fig, ax = plt.subplots(figsize=(6, 6.5))
ax.add_patch(plt.Rectangle((0, 0), Lx, Ly, fill=False, lw=2))
ax.plot(*SRC, 'r*', ms=16, label='Fonte')
for im, n in zip(IMAGES, SURFACES):
    ax.plot(*im, 'x', color='gray'); ax.annotate(n.replace('Parede ', ''), im, fontsize=8, color='gray')
ax.plot(*EAR_L, 'bo', label='Ouvido esq.'); ax.plot(*EAR_R, 'gs', label='Ouvido dir.')
ax.annotate('', xy=LIS + 0.8 * f, xytext=LIS, arrowprops=dict(arrowstyle='->'))
ax.set_xlim(-Lx - 0.5, 2 * Lx + 0.5); ax.set_ylim(-0.6 * Ly, 2.6 * Ly); ax.set_aspect('equal')
ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)'); ax.set_title('Planta, fonte e fontes-imagem'); ax.legend(loc='upper right')
plt.show()

## 3. Cauda reverberante descorrelacionada
Cada canal usa o reverberador de Schroeder do TP1 com $g_k = 10^{-3N_k/(F_s RT_{60})}$, mas com atrasos de *Comb* e *All-Pass* **diferentes** no canal direito. O decaimento e a energia são iguais nos dois canais, e as formas de onda diferem, o que reduz a correlação interaural. Cada cauda entra em $t_{mix}$ após a direta, com *fade-in* de 15 ms, e tem energia $E_{dir}\,10^{-DRR/10}$.

In [ ]:
def tail_ir(T_comb, T_ap, n_start, n_mix, n_ramp, E_target):
    L = int((1.5 * RT60 + 0.3) * Fs)
    delta = np.zeros(L); delta[0] = 1.0
    y = np.zeros(L)
    for T in T_comb:
        N = round(T * Fs / 1000); y += comb(delta, N, 10 ** (-3 * N / (Fs * RT60)))
    for T in T_ap:
        y = allpass(y, round(T * Fs / 1000), A_AP)
    k = np.arange(L); w = np.ones(L)
    w[k < n_mix - n_ramp] = 0
    r = (k >= n_mix - n_ramp) & (k < n_mix)
    w[r] = 0.5 * (1 - np.cos(np.pi * (k[r] - (n_mix - n_ramp)) / n_ramp))
    y = y * w; y *= np.sqrt(E_target / np.sum(y ** 2))
    h = np.zeros(n_start + L); h[n_start:] = y
    return h

d0 = np.linalg.norm(SRC - LIS); n_d0 = round(d0 / c * Fs)
n_mix, n_ramp = round(T_MIX * Fs), round(0.015 * Fs)
E_tail = (1 / d0 ** 2) * 10 ** (-DRR_DB / 10)
tail_L = tail_ir(T_COMB_L, T_AP_L, n_d0, n_mix, n_ramp, E_tail)
tail_R = tail_ir(T_COMB_R, T_AP_R, n_d0, n_mix, n_ramp, E_tail)
n_len = len(tail_L)

def early_ir(tab):
    h = np.zeros(n_len)
    for r in tab: h[r['n']] += r['g']
    return h

h_L = early_ir(tab_L) + tail_L
h_R = early_ir(tab_R) + tail_R

seg = slice(n_d0 + n_mix, n_len)
print(f'Correlação das caudas L/R: {np.corrcoef(tail_L[seg], tail_R[seg])[0, 1]:.3f}')

t = (np.arange(n_len) - n_d0) / Fs * 1000
fig, axs = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, h, nome in zip(axs, (h_L, h_R), ('esquerdo', 'direito')):
    ax.plot(t, 20 * np.log10(np.abs(h) + 1e-9), lw=0.5)
    ax.axvline(0, color='r', ls='--', label='direta'); ax.axvline(T_MIX * 1000, color='k', ls=':', label='t_mix')
    ax.set_xlim(-5, 300); ax.set_ylim(-110, 0); ax.set_xlabel('t após a direta (ms)'); ax.set_ylabel('dB')
    ax.set_title(f'Resposta impulsional, ouvido {nome}'); ax.legend()
plt.tight_layout(); plt.show()

fig, ax = plt.subplots()
m = (t > -2) & (t < 25)
ax.stem(t[m], h_L[m], linefmt='b-', markerfmt='bo', basefmt=' ', label='Esq.')
ax.stem(t[m], -h_R[m], linefmt='g-', markerfmt='gs', basefmt=' ', label='Dir. (espelhado)')
ax.set_xlabel('t após a direta (ms)'); ax.set_title('Direta e primeiras reflexões nos dois ouvidos'); ax.legend(); plt.show()

## 4. Saída estéreo

In [ ]:
x = read_wav(AUDIO_IN)
y = np.stack([fftconv(x, h_L), fftconv(x, h_R)], axis=1)
print(f'Pico antes de normalizar: {np.max(np.abs(y)):.3f}')
y_n = 0.99 * y / np.max(np.abs(y))          # fator comum: preserva a diferença interaural de nível
x_n = 0.99 * x / np.max(np.abs(x))
write_wav(f'{AUDIO_OUT}/DoisCanais_original.wav', x_n)
write_wav(f'{AUDIO_OUT}/DoisCanais_estereo.wav', y_n)
print('Pico final: %.3f   amostras >= 0.999: %d' % (np.max(np.abs(y_n)), np.sum(np.abs(y_n) >= 0.999)))
print('original'); display(Audio(x_n, rate=Fs))
print('estéreo (usar auscultadores)'); display(Audio(y_n.T, rate=Fs))

fig, axs = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
for ax, sig, nome in zip(axs, (x_n, y_n[:, 0], y_n[:, 1]), ('Original', 'Canal esq.', 'Canal dir.')):
    ax.plot(np.arange(len(sig)) / Fs, sig, lw=0.4); ax.set_ylabel(nome); ax.set_ylim(-1.05, 1.05)
axs[-1].set_xlabel('t (s)'); plt.tight_layout(); plt.show()

## 5. Validação
- $RT_{60}$ de cada canal por integração reversa, contra o alvo.
- ITD e ILD da onda direta com o ouvinte em várias posições.

In [ ]:
fig, ax = plt.subplots()
for h, nome in ((h_L, 'Esq.'), (h_R, 'Dir.')):
    e = h[n_d0:] ** 2; edc = np.cumsum(e[::-1])[::-1]; d = 10 * np.log10(edc / edc[0] + 1e-30); tt = np.arange(len(e)) / Fs
    mk = (d <= -5) & (d >= -35); rt = -60 / np.polyfit(tt[mk], d[mk], 1)[0]
    ax.plot(tt, d, label=f'{nome}: T30 = {rt:.2f} s')
ax.plot([0, RT60], [0, -60], 'k--', label=f'alvo RT60 = {RT60:.2f} s')
ax.set_ylim(-70, 3); ax.set_xlim(0, 1.5); ax.set_xlabel('t (s)'); ax.set_ylabel('EDC (dB)'); ax.legend(); plt.show()

POSICOES = [('Fundo, centro', (5.0, 6.5)), ('Meio da sala', (4.0, 5.0)), ('Perto da fonte', (2.5, 3.5)),
            ('Junto à parede x=Lx', (7.2, 4.0)), ('Canto traseiro', (7.0, 9.0))]
print(f"{'Posição':<22}{'d (m)':>7}{'ITD (amostras)':>16}{'ILD direta (dB)':>17}")
for nome, p in POSICOES:
    p = np.array(p); eL, eR = p + HEAD / 2 * left, p - HEAD / 2 * left
    tl, tr = taps(eL, +1)[0], taps(eR, -1)[0]
    print(f"{nome:<22}{np.linalg.norm(SRC - p):>7.2f}{tl['n'] - tr['n']:>16d}{20 * np.log10(tl['g'] / tr['g']):>17.2f}")

## 6. Análise crítica (esboço)
- **Sem HRTF:** a sombra da cabeça é uma lei de ganho simples ($1\pm\beta\cos\theta$) e os atrasos vêm só da geometria. Falta a filtragem do pavilhão, da cabeça e do tronco, dependente da frequência e da direção, pelo que a localização é mais fraca, há confusão frente/trás e tende a soar "dentro da cabeça".
- **Atrasos arredondados à amostra:** a resolução é 1 amostra ≈ 7,8 mm de percurso. Como o ITD máximo é 23 amostras, o erro de arredondamento (até 0,5 amostra) é relativamente significativo.
- **Descorrelação por atrasos distintos:** reduz a correlação das caudas, mas num campo difuso real a coerência entre ouvidos é alta nos graves e baixa nos agudos; aqui é baixa em todas as frequências.
- **Só 4 reflexões de 1.ª ordem** e plano horizontal: sem reflexões do chão e do teto (que chegam cedo e afetam a cor e a elevação percebida), nem ordens superiores além da cauda estatística.
- **Absorção igual em todas as paredes e frequências**, fonte omnidirecional, sala vazia.
- `RT60`, posições e `DRR_DB` são parâmetros assumidos até serem substituídos pelos do grupo.